# ResNet50 + Contrastive Regularization (Fine-tune)
This notebook fine-tunes ResNet50 with a contrastive regularizer.

Sections:
- 1) Setup and Imports
- 2) Dataset and Dataloaders
- 3) Model and Loss
- 4) Training Loop
- 5) Evaluation and Plots

Tips: Set `lambda_contrastive=0.0` to run CE-only baselines.


In [1]:
# 1) Setup and Imports
# Basic utilities and HDF5 visualization helper
import h5py
from dl_utils.utils.utils import viz_h5_structure

In [21]:
with h5py.File('../../datasets/imagenet_v5_rot_10k_fix_vector_a100_0-hierarchy_one_hot_encoding.h5', 'r') as f:
    viz_h5_structure(f)

'Dataset': labels; Shape: (10013,); dtype: uint8
'Dataset': labels_one_hot; Shape: (10013, 17); dtype: float64
'Dataset': normalized_primitive_uc_vector_a; Shape: (10013, 2); dtype: float64
'Dataset': normalized_primitive_uc_vector_b; Shape: (10013, 2); dtype: float64
'Dataset': normalized_rotation_angle; Shape: (10013,); dtype: float32
'Dataset': normalized_translation_start_point; Shape: (10013, 2); dtype: float64
'Dataset': reflection_labels; Shape: (10013,); dtype: uint8
'Dataset': reflection_one_hot; Shape: (10013, 2); dtype: float64
'Dataset': rotation_labels; Shape: (10013,); dtype: uint8
'Dataset': rotation_one_hot; Shape: (10013, 3); dtype: float64
'Dataset': shape; Shape: (10013,); dtype: uint8
'Dataset': shape_one_hot; Shape: (10013, 4); dtype: float64
'Dataset': similarity_vector; Shape: (10013, 7); dtype: float64
'Dataset': sub_rotation_labels; Shape: (10013,); dtype: uint8
'Dataset': sub_rotation_one_hot; Shape: (10013, 2); dtype: float64


In [22]:
# Additional imports for training and visualization
# Model builder, trainer, and the contrastive-regularized loss wrapper
import os, numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

from dl_utils.utils.dataset import hdf5_dataset, split_train_valid, viz_dataloader
from dl_utils.utils.utils import list_to_dict
from dl_utils.training.build_model import resnet50_
from dl_utils.training.trainer import Trainer, accuracy
from dl_utils.training.contrastive_loss import ContrastiveRegularizedLoss
import wandb

# Symmetry classes and helper to map label -> index
symmetry_classes = ['p1', 'p2', 'pm', 'pg', 'cm', 'pmm', 'pmg', 'pgg', 'cmm', 'p4', 'p4m', 'p4g', 'p3', 'p3m1', 'p31m', 'p6', 'p6m']
label_converter = list_to_dict(symmetry_classes)
n_classes = len(symmetry_classes)

In [29]:
# 2) Config: dataset paths, training hyperparameters, and contrastive settings
task_name = "ResNet50-fine_tune"
ds_path_info = {
    'imagenet': '../../datasets/imagenet_v5_rot_10k_fix_vector_a100_0-hierarchy_one_hot_encoding.h5',
    'atom': '../../datasets/atom_v5_rot_1m_fix_vector.h5',
    'noise': '../../datasets/noise_v5_rot_1m_fix_vector.h5',
    'viz_dataloader': False,  # set True to visualize few batches
}

training_specs = {
    'ds_size': 10_000,              # number of samples for each auxiliary set (noise/atom) and for imagenet subset build
    'batch_size': 256,
    'num_workers': 8,
    'learning_rate': 1e-3,
    'training_image_count': 2_000_000,  # for epochs calc: images_processed / len(train_ds)
    'validation_times': 10,
    'efficient_print': True,
    'model_path': '../../models/ResNet50/',
    'folder_name': 'default',        # or a custom name
    'device_ids': [4, 5, 6, 7, 8, 9],               # [0,1] if using DP
    'epoch_start': 0,
}

# W&B (optional)
wandb_specs = {}   # or populate as in your existing runs

# Contrastive settings
metadata_key = 'similarity_vector'    # HDF5 key under 'imagenet' folder that stores per-sample metadata
# lambda_contrastive = 0
lambda_contrastive = 1e-5
pos_threshold = 0.2
neg_threshold = 0.7
margin = 0.5
metadata_distance = 'cosine'     # 'l2' or 'cosine'
feature_layer = 'avgpool'    # resnet50_ has 'avgpool'
feature_norm = True


In [30]:
# 2) Dataset and Dataloaders
# Imagenet with metadata for training
imagenet_all = hdf5_dataset(
    ds_path_info['imagenet'],
    folder='imagenet',
    transform=transforms.ToTensor(),
    metadata_keys=metadata_key,
)
ratio = training_specs['ds_size'] * (1/0.8) / len(imagenet_all)
imagenet_sub, _ = split_train_valid(imagenet_all, ratio, seed=42)
train_ds, valid_ds = split_train_valid(imagenet_sub, 0.8, seed=42)

train_dl = DataLoader(train_ds, batch_size=training_specs['batch_size'], shuffle=True,  num_workers=training_specs['num_workers'])
valid_dl = DataLoader(valid_ds, batch_size=training_specs['batch_size'], shuffle=False, num_workers=training_specs['num_workers'])
viz_dataloader(valid_dl, label_converter=label_converter, title='imagenet - valid')

# Noise CV
noise_ds_all = hdf5_dataset(ds_path_info['noise'], folder='noise', transform=transforms.ToTensor())
ratio_noise = np.min((training_specs['ds_size'] / len(noise_ds_all), 1))
noise_ds, _ = split_train_valid(noise_ds_all, ratio_noise, seed=42)
noise_dl = DataLoader(noise_ds, batch_size=training_specs['batch_size'], shuffle=False, num_workers=training_specs['num_workers'])
viz_dataloader(noise_dl, label_converter=label_converter, title='noise - valid')

# Atom CV
atom_ds_all = hdf5_dataset(ds_path_info['atom'], folder='atom', transform=transforms.ToTensor())
ratio_atom = np.min((training_specs['ds_size'] / len(atom_ds_all), 1))
atom_ds, _ = split_train_valid(atom_ds_all, ratio_atom, seed=42)
atom_dl = DataLoader(atom_ds, batch_size=training_specs['batch_size'], shuffle=False, num_workers=training_specs['num_workers'])
viz_dataloader(atom_dl, label_converter=label_converter, title='atom - valid')

KeyError: "Unable to synchronously open object (object 'imagenet' doesn't exist)"

In [25]:
# 3) Model: ResNet50 backbone (with optional DataParallel)
device = torch.device(f"cuda:{training_specs['device_ids'][0]}" if torch.cuda.is_available() else "cpu")

model = resnet50_(in_channels=3, n_classes=n_classes)
model.load_state_dict(torch.load('../../models/ResNet50/03132025-ResNet50-benchmark-10m/epoch_23.pth', weights_only=True, map_location=torch.device('cpu')))

if len(training_specs['device_ids']) > 1:
    model = torch.nn.DataParallel(model, device_ids=training_specs['device_ids'])
model = model.to(device)

In [26]:
# 3b) Loss, optimizer, scheduler, and metrics
lr = training_specs['learning_rate']
epoch_start = training_specs.get('epoch_start', 0)
epochs = training_specs['training_image_count'] // len(train_ds) - epoch_start
valid_per_epochs = int(np.max((1, epochs / training_specs['validation_times'])))
early_stopping_patience = np.max((5, valid_per_epochs + 2))
efficient_print = training_specs['efficient_print']

base_ce = nn.CrossEntropyLoss()
loss_func = ContrastiveRegularizedLoss(
    base_criterion=base_ce,
    model=model,
    lambda_contrastive=lambda_contrastive,
    feature_layer=feature_layer,
    pos_threshold=pos_threshold,
    neg_threshold=neg_threshold,
    margin=margin,
    feature_norm=feature_norm,
    metadata_key=metadata_key,
    metadata_distance=metadata_distance,
)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, epochs=epochs, max_lr=lr, steps_per_epoch=len(train_dl)
)
metrics = [accuracy]


In [27]:
# 3c) Weights & Biases setup (optional)
NAME = task_name + '-dstaset_size=' + str(training_specs['ds_size'])
if wandb_specs:
    wandb.login()
    wandb.init(
        project=wandb_specs['project'], entity=wandb_specs['entity'],
        name=NAME, id=NAME, group=wandb_specs['group'],
        save_code=wandb_specs['save_code'], config=wandb_specs['config'], resume=wandb_specs['resume']
    )
    training_specs['wandb_record'] = True
else:
    training_specs['wandb_record'] = False

# 4) Training Loop and Validation
Runs the Trainer and logs per-epoch metrics including base CE and contrastive components (if enabled).


In [28]:
folder_name = training_specs['folder_name'] if training_specs['folder_name'] != 'default' else NAME
model_dir = os.path.join(training_specs['model_path'], folder_name)
os.makedirs(model_dir, exist_ok=True)

trainer = Trainer(
    model=model,
    loss_func=loss_func,
    optimizer=optimizer,
    metrics=metrics,
    scheduler=scheduler,
    device=device,
    save_per_epochs=valid_per_epochs,
    model_path=model_dir + '/',
    early_stopping_patience=early_stopping_patience,
    efficient_print=efficient_print,
)

history = trainer.train(
    train_dl=train_dl,
    epochs=epochs,
    epoch_start=epoch_start,
    valid_per_epochs=valid_per_epochs,
    valid_dl_list=[valid_dl, noise_dl, atom_dl],
    valid_dl_names=['', 'noise', 'atom'],
    wandb_record=training_specs['wandb_record'],
)

[0, 24, 48, 72, 96, 120, 144, 168, 192, 216, 240]
Epoch: 1/249


Train:   0%|          | 0/32 [00:01<?, ?it/s]


KeyError: Caught KeyError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/python3.11/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 50, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/python3.11/site-packages/torch/utils/data/dataset.py", line 418, in __getitems__
    return self.dataset.__getitems__([self.indices[idx] for idx in indices])  # type: ignore[attr-defined]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/py